# Clean QAChat Notebooks

This notebook finds executed Jupyter notebooks in the QAChat
repository and clears their execution counts and cell outputs.

Run the preview first. Set `CLEAN = True` only after checking
the reported notebooks.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path


def find_repository_root(start: Path | None = None) -> Path:
    """
    Find the QAChat repository root.

    Parameters
    ----------
    start : Path | None
        Directory from which to begin the search.

    Returns
    -------
    Path
        Absolute path to the QAChat repository root.

    Raises
    ------
    RuntimeError
        If the repository root cannot be found.

    """

    current = (start or Path.cwd()).resolve()

    for directory in (current, *current.parents):
        if (
            (directory / "qachat").is_dir()
            and (directory / "tests").is_dir()
        ):
            return directory

    raise RuntimeError("Could not locate the QAChat repository root.")


repository_root = find_repository_root()
repository_root


## Cleanup rules

Only execution state is removed. Markdown, code, notebook metadata,
and non-output cell metadata are left unchanged.


In [ ]:
CLEAN = True


## Find executed notebooks

This cell collects notebooks containing at least one code cell with
an execution count or output before anything is changed.


In [ ]:
def notebook_has_execution_state(path: Path) -> bool:
    """
    Check whether a notebook contains execution state.

    Parameters
    ----------
    path : Path
        Notebook to inspect.

    Returns
    -------
    bool
        Whether any code cell has an execution count or output.

    """

    with path.open("r", encoding="utf-8") as file:
        notebook = json.load(file)

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue

        if cell.get("execution_count") is not None:
            return True

        if cell.get("outputs"):
            return True

    return False


def find_executed_notebooks(root: Path) -> list[Path]:
    """
    Find notebooks containing execution state below a root.

    Parameters
    ----------
    root : Path
        Root directory to inspect.

    Returns
    -------
    list[Path]
        Executed notebooks sorted by path.

    """

    notebooks: list[Path] = []

    for path in root.rglob("*.ipynb"):
        if ".ipynb_checkpoints" in path.parts:
            continue

        if ".env" in path.parts:
            continue

        if notebook_has_execution_state(path):
            notebooks.append(path)

    return sorted(notebooks)


notebooks = find_executed_notebooks(repository_root)

print("Executed notebooks")
print("------------------")
for path in notebooks:
    print(path.relative_to(repository_root))

print("\nSummary")
print("-------")
print(f"{len(notebooks)} notebooks")


## Clear execution state

The default is a dry run. Change `CLEAN` to `True` in the cleanup
rules cell, rerun it, and then run this cell.


In [ ]:
def clear_notebook_execution_state(path: Path) -> bool:
    """
    Clear execution counts and outputs from one notebook.

    Parameters
    ----------
    path : Path
        Notebook to clean.

    Returns
    -------
    bool
        Whether the notebook was changed.

    """

    with path.open("r", encoding="utf-8") as file:
        notebook = json.load(file)

    changed = False

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue

        if cell.get("execution_count") is not None:
            cell["execution_count"] = None
            changed = True

        if cell.get("outputs"):
            cell["outputs"] = []
            changed = True

    if changed:
        with path.open("w", encoding="utf-8") as file:
            json.dump(notebook, file, ensure_ascii=False, indent=1)
            file.write("\n")

    return changed


if not CLEAN:
    print("Dry run only. Set CLEAN = True to clean notebooks.")
else:
    cleaned = 0

    for path in notebooks:
        if clear_notebook_execution_state(path):
            cleaned += 1

    print(f"Cleaned {cleaned} notebooks.")


## Verify cleanup

Run this final cell after cleaning to confirm that no notebook still
contains execution counts or outputs.


In [ ]:
remaining_notebooks = find_executed_notebooks(repository_root)

if remaining_notebooks:
    print("Executed notebooks remain:")
    for path in remaining_notebooks:
        print(path.relative_to(repository_root))
else:
    print("All notebooks are clean.")
